In [ ]:
# =====================================================================
# Sequential Trust-Region Optimization (TuRBO-inspired)
#   Outer loop: 3 macro-iterations of confidence-region relocation
#   Inner loop: RBF local surrogate + AGE-MOEA micro-search
#   Expected outcome: honest negative result — validates two-stage limit
# =====================================================================

import torch, numpy as np, math, time
from scipy.stats import gaussian_kde
from scipy.interpolate import RBFInterpolator
import matplotlib.pyplot as plt
from pymoo.core.problem import ElementwiseProblem
from pymoo.algorithms.moo.age import AGEMOEA
from pymoo.operators.crossover.sbx import SBX
from pymoo.operators.mutation.pm import PM
from pymoo.operators.sampling.rnd import FloatRandomSampling
from pymoo.termination import get_termination
from pymoo.optimize import minimize

device = torch.device('cuda'); print(f'Device: {device}')

# ============================================================
# 1. Physics Engine
# ============================================================
rx,ry,rz=10.0,10.0,3.0; xBs,yBs,zBs=10.0,-100.0,-10.0
E=8; dB=0.075; lam=0.075; L1=2; dref=abs(yBs)*1.5
Pd=40.0; Rth=0.2; Nd=-174.0; Bw=20e6; pm_val=0.2; nu=5
PB=10**(Pd/10)*1e-3; N0=10**(Nd/10)*1e-3*Bw

def gen_rwp_traj(sim_time,dt=10,nu=5,rng=None):
    if rng is None: rng=np.random
    rs=[0.0,rx,0.0,ry]; hc=np.array([rx/2,ry/2]); hr=rx/3.0
    p_t=0.6;p_s=0.3;tau_h=1.5;tau_n=0.1;v_h=0.2;v_n=1.0;g_h=0.6;g_n=0.2
    ts=int(sim_time/dt)
    def gt(p):
        t=hc+(rng.rand(2)-0.5)*2*hr if rng.rand()<g_h else np.array([rs[0]+rng.rand()*(rs[1]-rs[0]),rs[2]+rng.rand()*(rs[3]-rs[2])])
        t[0]=np.clip(t[0],rs[0],rs[1]);t[1]=np.clip(t[1],rs[2],rs[3]);return t
    def ta(r):return tau_h if r=='hot'else tau_n
    def sp(r):return v_h if r=='hot'else v_n
    uh=1.5+0.5*rng.rand(nu);pos=np.zeros((nu,ts,2))
    up=np.zeros((nu,2));ur=[None]*nu;us=[None]*nu;ut_=np.zeros((nu,2));ud=np.zeros((nu,2));usp=np.zeros(nu);uprem=np.zeros(nu)
    for i in range(nu):
        up[i]=[rs[0]+rng.rand()*(rs[1]-rs[0]),rs[2]+rng.rand()*(rs[3]-rs[2])];d2h=np.linalg.norm(up[i]-hc);ur[i]='hot'if d2h<=hr else'normal'
        if rng.rand()<p_t:us[i]='transfer';ut_[i]=gt(up[i]);dv=ut_[i]-up[i];ud[i]=dv/np.linalg.norm(dv);usp[i]=sp(ur[i])
        else:us[i]='pause';uprem[i]=rng.exponential(ta(ur[i]))
        pos[i,0,:]=up[i]
    for step in range(1,ts):
        for i in range(nu):
            if us[i]=='pause':
                uprem[i]-=dt;pos[i,step,:]=up[i]
                if uprem[i]<=0:us[i]='transfer';ut_[i]=gt(up[i]);dv=ut_[i]-up[i];ud[i]=dv/np.linalg.norm(dv);usp[i]=sp(ur[i])
            else:
                md=usp[i]*dt;pd=ut_[i]-up[i]
                if np.linalg.norm(pd)<=md:
                    up[i]=ut_[i].copy();d2h=np.linalg.norm(up[i]-hc);ur[i]='hot'if d2h<=hr else'normal'
                    if rng.rand()<p_s:us[i]='pause';uprem[i]=rng.exponential(ta(ur[i]))
                    else:ut_[i]=gt(up[i]);dv=ut_[i]-up[i];ud[i]=dv/np.linalg.norm(dv);usp[i]=sp(ur[i])
                else:up[i]=up[i]+ud[i]*md
                pos[i,step,:]=up[i]
    pts=np.zeros((nu*ts,3));idx=0
    for u in range(nu):
        for s in range(ts):pts[idx]=[pos[u,s,0],pos[u,s,1],uh[u]];idx+=1
    return pts

GR=200; Z_HS=[1.5,1.625,1.75,1.875,2.0]; N_Z=len(Z_HS)
xv=torch.linspace(0,rx,GR,device=device); yv=torch.linspace(0,ry,GR,device=device)
Xg,Yg=torch.meshgrid(xv,yv,indexing='ij'); xyf=torch.stack([Xg.flatten(),Yg.flatten()],dim=1)
gl=[]; [gl.append(torch.stack([Xg.flatten(),Yg.flatten(),torch.full_like(Xg.flatten(),zh)],dim=1)) for zh in Z_HS]
gl=torch.cat(gl,dim=0); Ngrid=gl.shape[0]
np.random.seed(777); et=gen_rwp_traj(sim_time=100000,dt=10)
kde=gaussian_kde(et[:,:2].T,bw_method='scott')
wxy=kde(xyf.cpu().numpy().T).flatten(); wxy=wxy/wxy.sum()
gw=torch.tensor(np.tile(wxy,N_Z),dtype=torch.float32,device=device); gw=gw/gw.sum()
np.random.seed(42)
_ps=torch.tensor(2*np.pi*np.random.rand(Ngrid),dtype=torch.float32,device=device)
_tt=torch.tensor(-np.pi+2*np.pi*np.random.rand(Ngrid),dtype=torch.float32,device=device)
_eta=torch.tensor(10**((-15+5*np.random.rand(Ngrid))/10),dtype=torch.float32,device=device)
_n_ant=torch.arange(E,dtype=torch.float32,device=device)
_dy_BS=torch.tensor(0.0-yBs,device=device); _v1c=lam/(4*math.pi); _v1pc=-(2*math.pi/lam); _oE=1/math.sqrt(E)

def phys_chan(wp,lc):
    if not isinstance(wp,torch.Tensor): wp=torch.tensor(wp,dtype=torch.float32,device=device)
    if wp.dim()==1: wp=wp.unsqueeze(0)
    Bn=wp.shape[0]; xc,zc,Lx,Lz=wp[:,0],wp[:,1],wp[:,2],wp[:,3]
    xu,yu,zu=lc[:,0],lc[:,1],lc[:,2]
    dx=xc-xBs; dy=_dy_BS; dz=zc-zBs
    Rw=torch.sqrt(dx**2+dy**2+dz**2); th=torch.atan2(dy,dx); ph=torch.acos(dz/Rw)
    kx=torch.sin(ph)*torch.cos(th); kz=torch.cos(ph)
    x1=xc-Lx/2;z1=zc-Lz/2;x2=xc-Lx/2;z2=zc+Lz/2;x3=xc+Lx/2;z3=zc-Lz/2;x4=xc+Lx/2;z4=zc+Lz/2
    def rd(xs,zs):
        ddx=xs.unsqueeze(1)-xBs;ddy=_dy_BS;ddz=zs.unsqueeze(1)-zBs
        L=torch.sqrt(ddx**2+ddy**2+ddz**2);return ddx/L,ddy/L,ddz/L
    ux1,_,uz1=rd(x1,z1);ux2,_,uz2=rd(x2,z2);ux3,_,uz3=rd(x3,z3);ux4,_,uz4=rd(x4,z4)
    dux=xu-xBs;duy=yu-yBs;duz=zu-zBs;Lu=torch.sqrt(dux**2+duy**2+duz**2)
    uux=dux/Lu;uuz=duz/Lu
    ua=torch.stack([ux1,ux2,ux3,ux4],0);uz=torch.stack([uz1,uz2,uz3,uz4],0)
    umin=ua.min(0).values;umax=ua.max(0).values;zmin=uz.min(0).values;zmax=uz.max(0).values
    b=1000.0
    ix=torch.sigmoid(b*(uux-umin))*torch.sigmoid(b*(umax-uux))
    iz=torch.sigmoid(b*(uuz-zmin))*torch.sigmoid(b*(zmax-uuz))
    il=ix*iz
    d2x=xu-xc.unsqueeze(1);d2y=yu;d2z=zu-zc.unsqueeze(1)
    Ru=torch.sqrt(d2x**2+d2y**2+d2z**2);t1=d2x/Ru;t2=d2z/Ru
    ax=(1-il)*(kx.unsqueeze(1)-t1);az=(1-il)*(kz.unsqueeze(1)-t2)
    sx=torch.sinc((math.pi/lam)*Lx.unsqueeze(1)*ax);sz=torch.sinc((math.pi/lam)*Lz.unsqueeze(1)*az)
    p1=(2*math.pi/lam)*dB*_n_ant*torch.sin(th).unsqueeze(1)
    a1=_oE*torch.complex(torch.cos(p1),torch.sin(p1))
    v1m=_v1c/Rw;v1p=_v1pc*Rw
    v1=torch.complex(v1m*torch.cos(v1p),v1m*torch.sin(v1p));H1=v1.unsqueeze(1)*a1.conj()
    p2=(2*math.pi/lam)*dB*_n_ant.unsqueeze(0)*torch.sin(_tt).unsqueeze(1)
    a2=_oE*torch.complex(torch.cos(p2),torch.sin(p2))
    v2m=_eta*(lam/(4*math.pi*dref));v2=torch.complex(v2m*torch.cos(_ps),v2m*torch.sin(_ps))
    H2=v2.unsqueeze(1)*a2.conj().unsqueeze(0)
    Ht=math.sqrt(E/L1)*(H1.unsqueeze(1)+H2)
    fm=(Lx.unsqueeze(1)*Lz.unsqueeze(1)*sx*sz)/(lam*Ru)
    fp=(-(2*math.pi/lam)*(kx*xc+kz*zc)-(math.pi/2))
    fc=torch.complex(fm*torch.cos(fp.unsqueeze(1)),fm*torch.sin(fp.unsqueeze(1)))
    return Ht*fc.unsqueeze(2)

@torch.no_grad()
def physics_outage(X_np):
    out=[]
    for i in range(0,len(X_np),200):
        xb=X_np[i:i+200]; th=torch.tensor(xb,dtype=torch.float32,device=device)
        He=phys_chan(th,gl); Hw=torch.sum(He,dim=2)/math.sqrt(E)
        dp=(torch.abs(Hw)**2)*pm_val*PB; it=(nu-1)*dp; sr=dp/(it+N0)
        xc_,zc_=th[:,0],th[:,1]; ab=math.pi/3.0; Ses=torch.zeros_like(sr)
        rn=torch.sqrt((gl[:,0].unsqueeze(0)-xc_.unsqueeze(1))**2+(gl[:,2].unsqueeze(0)-zc_.unsqueeze(1))**2+gl[:,1].unsqueeze(0)**2+1e-12)
        for a in [0.0,math.pi/2,math.pi,3*math.pi/2]:
            dot=torch.abs((gl[:,0].unsqueeze(0)-xc_.unsqueeze(1))*math.cos(a)+gl[:,1].unsqueeze(0)*math.sin(a))
            cp=dot/rn; ph=torch.acos(torch.clamp(cp,0,1))
            Se=(math.pi-torch.clamp(2*ab-ph,min=0))/math.pi; Ses=Ses+torch.clamp(Se,1/3,1)
        sr=((Ses/4)*dp)/((Ses/4)*it+N0)
        bits=(torch.log2(1+sr)<Rth).float()
        out.append(torch.sum(bits*gw,dim=1).cpu().numpy()); torch.cuda.empty_cache()
    return np.concatenate(out)

print('Physics engine ready.')

# ============================================================
# 2. Initial State (from two-stage warm-start)
# ============================================================
lb=np.array([0.2,0.2,0.1,0.1]); ub=np.array([9.8,2.8,9.8,2.8])
curr_center = np.array([5.263, 1.300, 9.008, 0.184])
curr_radius = 0.05 * curr_center
best_phys_area = curr_center[2] * curr_center[3]
best_phys_outage = physics_outage(curr_center.reshape(1,-1))[0]

print(f'\nInitial state:')
print(f'  Center: [{curr_center[0]:.3f},{curr_center[1]:.3f},{curr_center[2]:.3f},{curr_center[3]:.3f}]')
print(f'  Area={best_phys_area:.4f} m²  Outage={best_phys_outage*100:.2f}%')

# ============================================================
# 3. Trust-Region Loop
# ============================================================
N_MACRO = 3
history = [{'area':best_phys_area, 'outage':best_phys_outage, 'center':curr_center.copy(), 'event':'start'}]

for k in range(1, N_MACRO+1):
    print(f'\n{"="*50}')
    print(f' Trust-Region Iteration {k}/{N_MACRO}')
    print(f' Radius: [{curr_radius[0]:.3f},{curr_radius[1]:.3f},{curr_radius[2]:.3f},{curr_radius[3]:.3f}]')
    print(f'{"="*50}')

    # Compute local bounds
    xl_local = np.clip(curr_center - curr_radius, lb, ub)
    xu_local = np.clip(curr_center + curr_radius, lb, ub)

    # Sample 100 points inside trust region, select balanced 15+15
    np.random.seed(1000+k)
    cand_X = np.random.uniform(xl_local, xu_local, size=(100,4))
    cand_Y = physics_outage(cand_X)
    safe = cand_Y <= 0.10
    unsafe = cand_Y > 0.10
    dists = np.linalg.norm((cand_X - curr_center)/(curr_center+1e-6), axis=1)
    if safe.sum() >= 15 and unsafe.sum() >= 15:
        si = np.where(safe)[0][np.argsort(dists[safe])[:15]]
        ui = np.where(unsafe)[0][np.argsort(dists[unsafe])[:15]]
        local_X = np.vstack([cand_X[si], cand_X[ui]])
        local_Y = np.concatenate([cand_Y[si], cand_Y[ui]])
        print(f'  Balanced: {len(si)} safe + {len(ui)} unsafe')
    else:
        idx = np.argsort(dists)[:30]
        local_X = cand_X[idx]; local_Y = cand_Y[idx]
        print(f'  Fallback: {len(idx)} nearest (safe={safe.sum()}/{len(cand_Y)})')

    # RBF surrogate
    rbf = RBFInterpolator(local_X, local_Y, kernel='thin_plate_spline')

    # Local AGE-MOEA on RBF surrogate
    class LocalProblem(ElementwiseProblem):
        def __init__(self):
            super().__init__(n_var=4, n_obj=2, xl=xl_local, xu=xu_local)
        def _evaluate(self, x, out, *a, **kw):
            pred = float(rbf(x.reshape(1,-1))[0])
            out["F"] = [x[2]*x[3], pred]

    local_algo = AGEMOEA(pop_size=30, sampling=FloatRandomSampling(),
        crossover=SBX(prob=0.9, eta=15), mutation=PM(prob=0.35, eta=15))
    res_loc = minimize(LocalProblem(), local_algo, get_termination('n_gen', 20), seed=42, verbose=False)

    # Extract best feasible candidate from local search
    F_loc = res_loc.F; X_loc = res_loc.X
    feas_loc = F_loc[:,1] <= 0.10
    if feas_loc.any():
        bi = np.argmin(F_loc[feas_loc,0])
        cand_x = X_loc[feas_loc][bi]
    else:
        cand_x = X_loc[np.argmin(F_loc[:,0])]  # fallback: smallest area

    # Physics verification
    cand_out = physics_outage(cand_x.reshape(1,-1))[0]
    cand_area = cand_x[2]*cand_x[3]

    # Trust-region logic
    if cand_out <= 0.10 and cand_area < best_phys_area:
        print(f'  [SUCCESS] Area: {best_phys_area:.4f} -> {cand_area:.4f} m²')
        best_phys_area = cand_area
        best_phys_outage = cand_out
        curr_center = cand_x
        curr_radius = 0.8 * curr_radius
        history.append({'area':cand_area, 'outage':cand_out, 'center':cand_x.copy(), 'event':'success'})
    else:
        violation = 'outage' if cand_out > 0.10 else 'area'
        print(f'  [FAILURE] {violation} {cand_out*100:.2f}% / {cand_area:.4f} m² — shrinking radius')
        curr_radius = 0.5 * curr_radius
        history.append({'area':best_phys_area, 'outage':best_phys_outage, 'center':curr_center.copy(), 'event':'failure'})

# ============================================================
# 4. Summary
# ============================================================
print('\n' + '='*60)
print('TRUST-REGION SEQUENCE')
print('='*60)
for i,h in enumerate(history):
    print(f'  {i}: [{h["center"][0]:.3f},{h["center"][1]:.3f},{h["center"][2]:.3f},{h["center"][3]:.3f}] area={h["area"]:.4f} out={h["outage"]*100:.2f}% [{h["event"]}]')

fig,ax=plt.subplots(figsize=(8,3))
areas=[h['area'] for h in history]
ax.bar(range(len(areas)),areas,color=['gray','red','red','red'])
ax.axhline(y=1.64,color='green',ls='--',label=f'NSGA-II ref: 1.64 m²')
ax.set_xticks(range(len(areas)))
ax.set_xticklabels([h['event'] for h in history])
ax.set_ylabel('Area [m²]'); ax.legend()
ax.set_title('Trust-Region: honest negative result')
plt.tight_layout(); plt.savefig('trust_region.png',dpi=150); plt.show()

from IPython.display import FileLink,display
display(FileLink('trust_region.png'))